# Rental Rights Test Collection

This notebook builds the ground-truth question set used to evaluate the Victorian Rental Rights RAG system. Questions are linked to the knowledge-base chunks that contain the information needed to answer them.

In [19]:
from pathlib import Path
import pandas as pd

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR / "data" / "processed"
elif (CURRENT_DIR.parent / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR.parent / "data" / "processed"
else:
    raise FileNotFoundError("Could not locate data/processed folder")

kb_path = DATA_DIR / "rental_kb_chunks.jsonl"

kb_df = pd.read_json(kb_path, lines=True)

print("Knowledge-base chunks loaded:", len(kb_df))
display(kb_df.head())

Knowledge-base chunks loaded: 32


,chunk_id,document_id,document_name,page,text,source_file
0,RG_P04,RG,Renters Guide,4,Introduction\nVictoria has some of the stronge...,Renters Guide.pdf
1,RG_P07,RG,Renters Guide,7,Before you apply\nDocuments and information yo...,Renters Guide.pdf
2,RG_P08,RG,Renters Guide,8,More information: consumer.vic.gov.au/unlawful...,Renters Guide.pdf
3,RG_P09,RG,Renters Guide,9,Read through and complete the rental applicati...,Renters Guide.pdf
4,RG_P11,RG,Renters Guide,11,Communicating with your rental provider\nYou c...,Renters Guide.pdf


## 1. Review Knowledge-Base Coverage

Review the available chunks and their main topics before creating the evaluation questions.

In [20]:
# Create a compact overview of the available KB chunks

chunk_overview = kb_df[
    ["chunk_id", "document_name", "page", "text"]
].copy()

chunk_overview["preview"] = (
    chunk_overview["text"]
    .str.replace("\n", " ", regex=False)
    .str.slice(0, 180)
)

display(
    chunk_overview[
        ["chunk_id", "document_name", "page", "preview"]
    ]
)

,chunk_id,document_name,page,preview
0,RG_P04,Renters Guide,4,Introduction Victoria has some of the stronges...
1,RG_P07,Renters Guide,7,Before you apply Documents and information you...
2,RG_P08,Renters Guide,8,More information: consumer.vic.gov.au/unlawful...
3,RG_P09,Renters Guide,9,Read through and complete the rental applicati...
4,RG_P11,Renters Guide,11,Communicating with your rental provider You ca...
5,RG_P12,Renters Guide,12,Rental minimum standards guide There are 15 ca...
6,RG_P13,Renters Guide,13,Learn more about each of the minimum standards...
7,RG_P14,Renters Guide,14,Kitchen The property must have a kitchen with:...
8,RG_P15,Renters Guide,15,Locks The property’s external entry doors must...
9,RG_P16,Renters Guide,16,Ventilation Rental properties must have adequa...


## 2. Create Known Questions

Known questions have an answer that is directly available in one or more knowledge-base chunks. The first set covers a range of rental topics rather than focusing on one area.

In [21]:
# Initial Known question set

known_questions = [
    {
        "question_id": "K01",
        "question": "Can a rental provider ask me about disputes with a previous landlord when I apply?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P07"]
    },
    {
        "question_id": "K02",
        "question": "Can a rental provider accept an offer above the advertised rent?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P08"]
    },
    {
        "question_id": "K03",
        "question": "Does a Victorian rental property need to have a fixed heater?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P13", "MS_P01"]
    },
    {
        "question_id": "K04",
        "question": "How long do I have to return my completed condition report after moving in?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P17", "RG_P19"]
    },
    {
        "question_id": "K05",
        "question": "How much rent can I be asked to pay in advance?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P21"]
    },
    {
        "question_id": "K06",
        "question": "How much notice must I receive before my rent is increased?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P22"]
    },
    {
        "question_id": "K07",
        "question": "Is a broken toilet considered an urgent repair?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P24"]
    },
    {
        "question_id": "K08",
        "question": "What happens if my rental provider wants to refuse my request to keep a pet?",
        "question_type": "Known",
        "relevant_chunk_ids": ["RG_P28"]
    }
]

test_df = pd.DataFrame(known_questions)

display(test_df)

,question_id,question,question_type,relevant_chunk_ids
0,K01,Can a rental provider ask me about disputes wi...,Known,[RG_P07]
1,K02,Can a rental provider accept an offer above th...,Known,[RG_P08]
2,K03,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]"
3,K04,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]"
4,K05,How much rent can I be asked to pay in advance?,Known,[RG_P21]
5,K06,How much notice must I receive before my rent ...,Known,[RG_P22]
6,K07,Is a broken toilet considered an urgent repair?,Known,[RG_P24]
7,K08,What happens if my rental provider wants to re...,Known,[RG_P28]


## 3. Add Expected Answers

Record a concise ground-truth answer for each question based only on the information contained in the knowledge base.

In [22]:
# Add expected answers for the Known questions

expected_answers = {
    "K01": (
        "No. A rental provider or agent cannot ask whether you have taken legal action "
        "or had a dispute with a previous rental provider."
    ),
    "K02": (
        "No. Rental providers and agents cannot ask for, invite or accept offers of rent "
        "higher than the advertised price."
    ),
    "K03": (
        "Yes. Rental properties must have a fixed heater in good working order in the main "
        "living area. For rental agreements starting from 29 March 2023, the heater must "
        "also be energy efficient."
    ),
    "K04": (
        "You must return one completed and signed copy of the condition report within "
        "5 business days of moving in."
    ),
    "K05": (
        "If rent is paid weekly, you can be asked for up to 2 weeks' rent in advance. "
        "If rent is paid monthly and the weekly rent is $900 or less, the maximum is one "
        "month's rent. Different rules apply where the weekly rent is above $900."
    ),
    "K06": (
        "The rental provider must give you a Notice of rent increase at least 90 days "
        "before the rent increase takes effect."
    ),
    "K07": (
        "Yes. A blocked or broken toilet system is legally defined as an urgent repair "
        "and must be repaired immediately."
    ),
    "K08": (
        "If the rental provider wants to refuse consent for a pet, they must apply to "
        "VCAT within 14 days. VCAT then decides whether refusing consent is reasonable."
    )
}

test_df["expected_answer"] = test_df["question_id"].map(expected_answers)

display(test_df)

,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,K01,Can a rental provider ask me about disputes wi...,Known,[RG_P07],No. A rental provider or agent cannot ask whet...
1,K02,Can a rental provider accept an offer above th...,Known,[RG_P08],No. Rental providers and agents cannot ask for...
2,K03,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]",Yes. Rental properties must have a fixed heate...
3,K04,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]",You must return one completed and signed copy ...
4,K05,How much rent can I be asked to pay in advance?,Known,[RG_P21],"If rent is paid weekly, you can be asked for u..."
5,K06,How much notice must I receive before my rent ...,Known,[RG_P22],The rental provider must give you a Notice of ...
6,K07,Is a broken toilet considered an urgent repair?,Known,[RG_P24],Yes. A blocked or broken toilet system is lega...
7,K08,What happens if my rental provider wants to re...,Known,[RG_P28],If the rental provider wants to refuse consent...


## 4. Create Inferred Questions

Inferred questions require information from multiple knowledge-base chunks to produce a complete answer.

In [23]:
# Initial Inferred question set

inferred_questions = [
    {
        "question_id": "I01",
        "question": "My rental has no working fixed heater. Is this considered an urgent repair, and what can I do if the rental provider does not respond?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P13", "MS_P01", "RG_P24", "RG_P25"]
    },
    {
        "question_id": "I02",
        "question": "My rent is being increased and I think the new amount is too high. How much notice should I receive and what can I do to challenge it?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P22", "RG_P23"]
    },
    {
        "question_id": "I03",
        "question": "My rental provider wants to claim my bond for damage that was already there when I moved in. What information could help me dispute the claim?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P17", "RG_P34", "RG_P35"]
    },
    {
        "question_id": "I04",
        "question": "Can my rental provider evict me because I have tried to exercise my rental rights, and what do they normally need to end the agreement?",
        "question_type": "Inferred",
        "relevant_chunk_ids": ["RG_P32", "RG_P33"]
    }
]

inferred_df = pd.DataFrame(inferred_questions)

display(inferred_df)

,question_id,question,question_type,relevant_chunk_ids
0,I01,My rental has no working fixed heater. Is this...,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]"
1,I02,My rent is being increased and I think the new...,Inferred,"[RG_P22, RG_P23]"
2,I03,My rental provider wants to claim my bond for ...,Inferred,"[RG_P17, RG_P34, RG_P35]"
3,I04,Can my rental provider evict me because I have...,Inferred,"[RG_P32, RG_P33]"


## 5. Add Expected Answers for Inferred Questions

Create ground-truth answers by combining the information contained across the relevant knowledge-base chunks.

In [24]:
# Add expected answers for the Inferred questions

inferred_answers = {
    "I01": (
        "Yes. A rental property must have a working fixed heater in the main living area, "
        "and a property that does not meet the minimum standards is considered an urgent repair. "
        "The rental provider must arrange the repair immediately. If they do not respond, you can "
        "organise and pay for the urgent repair yourself if it costs no more than $2500. The rental "
        "provider must reimburse you within 7 days, and if they do not, you can apply to RDRV."
    ),
    "I02": (
        "You must receive a Notice of rent increase at least 90 days before the increase takes effect. "
        "If you believe the increase is too high, you can ask Consumer Affairs Victoria for a free rent "
        "assessment within 30 days of receiving the notice. If you still cannot agree on the rent after "
        "the assessment, you can apply to RDRV or VCAT."
    ),
    "I03": (
        "Your condition report can help show that the damage was already present when you moved in, "
        "especially if you recorded the damage and took photos. The exit condition report can then be "
        "compared with the original condition report. A rental provider can claim the bond for damage "
        "caused by you or your visitors, but not for fair wear and tear. If you disagree with the bond "
        "claim, you can dispute it through RDRV."
    ),
    "I04": (
        "No. A rental provider cannot evict you because you have used or intend to use your rental rights. "
        "To end the rental agreement, they generally need a valid reason, the correct written Notice to "
        "vacate, and the required amount of notice. The notice period depends on the reason, although the "
        "guide states that in most instances it is 90 days."
    )
}

inferred_df["expected_answer"] = inferred_df["question_id"].map(inferred_answers)

display(inferred_df)

,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,I01,My rental has no working fixed heater. Is this...,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]",Yes. A rental property must have a working fix...
1,I02,My rent is being increased and I think the new...,Inferred,"[RG_P22, RG_P23]",You must receive a Notice of rent increase at ...
2,I03,My rental provider wants to claim my bond for ...,Inferred,"[RG_P17, RG_P34, RG_P35]",Your condition report can help show that the d...
3,I04,Can my rental provider evict me because I have...,Inferred,"[RG_P32, RG_P33]",No. A rental provider cannot evict you because...


## 6. Create Out-of-KB Questions

Out-of-KB questions test whether the RAG system can recognise when the available knowledge base does not contain enough information to provide a supported answer.

In [25]:
# Initial Out-of-KB question set

out_of_kb_questions = [
    {
        "question_id": "O01",
        "question": "Can I sublet my rental property to another person without my rental provider's permission?",
        "question_type": "Out-of-KB",
        "relevant_chunk_ids": [],
        "expected_answer": "The current knowledge base does not contain enough information to answer this question."
    },
    {
        "question_id": "O02",
        "question": "How much does it cost to make a rental application to VCAT?",
        "question_type": "Out-of-KB",
        "relevant_chunk_ids": [],
        "expected_answer": "The current knowledge base does not contain enough information to answer this question."
    },
    {
        "question_id": "O03",
        "question": "Can my rental provider install security cameras inside my rental property?",
        "question_type": "Out-of-KB",
        "relevant_chunk_ids": [],
        "expected_answer": "The current knowledge base does not contain enough information to answer this question."
    }
]

out_of_kb_df = pd.DataFrame(out_of_kb_questions)

display(out_of_kb_df)

,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,O01,Can I sublet my rental property to another per...,Out-of-KB,[],The current knowledge base does not contain en...
1,O02,How much does it cost to make a rental applica...,Out-of-KB,[],The current knowledge base does not contain en...
2,O03,Can my rental provider install security camera...,Out-of-KB,[],The current knowledge base does not contain en...


## 7. Finalise the Version 1 Test Collection

Combine the Known, Inferred and Out-of-KB questions, check that all labelled relevant chunks exist in the knowledge base, and save the test collection for later evaluation.

In [26]:
# Combine the three question types

test_collection_df = pd.concat(
    [test_df, inferred_df, out_of_kb_df],
    ignore_index=True
)

# Check that all labelled relevant chunks exist in the KB
valid_chunk_ids = set(kb_df["chunk_id"])

invalid_chunks = []

for _, row in test_collection_df.iterrows():
    for chunk_id in row["relevant_chunk_ids"]:
        if chunk_id not in valid_chunk_ids:
            invalid_chunks.append((row["question_id"], chunk_id))

print("Total questions:", len(test_collection_df))
print("\nQuestion types:")
print(test_collection_df["question_type"].value_counts())

print("\nInvalid relevant chunk IDs:", invalid_chunks)

display(test_collection_df)

Total questions: 15

Question types:
question_type
Known        8
Inferred     4
Out-of-KB    3
Name: count, dtype: int64

Invalid relevant chunk IDs: []


,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,K01,Can a rental provider ask me about disputes wi...,Known,[RG_P07],No. A rental provider or agent cannot ask whet...
1,K02,Can a rental provider accept an offer above th...,Known,[RG_P08],No. Rental providers and agents cannot ask for...
2,K03,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]",Yes. Rental properties must have a fixed heate...
3,K04,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]",You must return one completed and signed copy ...
4,K05,How much rent can I be asked to pay in advance?,Known,[RG_P21],"If rent is paid weekly, you can be asked for u..."
5,K06,How much notice must I receive before my rent ...,Known,[RG_P22],The rental provider must give you a Notice of ...
6,K07,Is a broken toilet considered an urgent repair?,Known,[RG_P24],Yes. A blocked or broken toilet system is lega...
7,K08,What happens if my rental provider wants to re...,Known,[RG_P28],If the rental provider wants to refuse consent...
8,I01,My rental has no working fixed heater. Is this...,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]",Yes. A rental property must have a working fix...
9,I02,My rent is being increased and I think the new...,Inferred,"[RG_P22, RG_P23]",You must receive a Notice of rent increase at ...


In [27]:
# Save the Version 1 test collection

test_output_path = DATA_DIR / "rental_test_collection_v1.jsonl"

test_collection_df.to_json(
    test_output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved test collection to:")
print(test_output_path)

print("\nQuestions saved:", len(test_collection_df))

Saved test collection to:
/Users/seanrichards/Documents/University/DS Case Studies/Project/data/processed/rental_test_collection_v1.jsonl

Questions saved: 15


## 8. Create Question Variants

To test how robust the RAG system is to different ways of asking the same question, create four variations of each original test question.

Each variation keeps the same underlying information need, but changes how the question is expressed:

- **V1 – Conversational:** natural wording that a renter might use
- **V2 – Short / informal:** shorter and less structured wording
- **V3 – Grammar / spelling:** small realistic grammar or spelling mistakes
- **V4 – Alternative structure:** the same question expressed using a different sentence structure

The original question remains unchanged. These variants will later inherit the same question type, relevant chunks and expected answer as their base question.


In [28]:
# Four wording variants for each original question

question_variants = {

    # -------------------------
    # Known Questions
    # -------------------------

    "K01": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "When I apply for a rental, can they ask if I had any disputes with my old landlord?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Can they ask about problems I had with a previous landlord?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "Can a landlord ask if ive had disputes with a previous rental provider?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "If I had a dispute with a previous rental provider, am I required to tell a new one when applying?"
        }
    ],

    "K02": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "If I offer more than the advertised rent, is the rental provider allowed to accept it?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Can a landlord take an offer above the listed rent?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "Can they accept more rent then what was advertised?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "What happens if someone offers the rental provider more than the advertised rental price?"
        }
    ],

    "K03": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "Does the place I'm renting in Victoria have to come with a fixed heater?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Does my rental need a fixed heater?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "Dose a Victorian rental have to have a fixed heater?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "Is having a fixed heater one of the requirements for rental properties in Victoria?"
        }
    ],

    "K04": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "I've just moved in. How long do I have to give back the completed condition report?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "When do I need to return the condition report?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "How long do i have to send back my condition report after moving in?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "After moving into a rental, what is the deadline for returning the completed condition report?"
        }
    ],

    "K05": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "How much rent is my rental provider allowed to ask me to pay upfront?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "How much rent can they ask for in advance?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "How much rent can i be made to pay upfront in advance?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "Are there limits on the amount of rent that can be charged before I move in?"
        }
    ],

    "K06": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "My landlord wants to increase my rent. How much notice do they have to give me?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "How much warning do I get before a rent increase?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "How much notice do they need to give before putting my rent up?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "Before a rent increase can take effect, how far in advance must I be notified?"
        }
    ],

    "K07": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "My toilet has broken. Does that count as an urgent repair?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Broken toilet - is that urgent?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "My toilet is broke is this classed as a urgent repair?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "If the toilet in my rental stops working, is it treated as an urgent repair?"
        }
    ],

    "K08": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "I asked to keep a pet and my rental provider wants to say no. What happens next?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "What happens if my landlord says no to my pet?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "My rental provider refused my pet request what happens now?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "If a rental provider does not want to allow a pet, what do they need to do?"
        }
    ],

    # -------------------------
    # Inferred Questions
    # -------------------------

    "I01": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "The fixed heater in my rental doesn't work. Is that urgent, and what can I do if the landlord doesn't organise a repair?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "No working heater. Is it urgent and what can I do if they don't fix it?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "My fixed heater isnt working is this urgent and what do i do if the landlord doesnt respond?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "What options do I have if my rental has no working fixed heater and the rental provider does not respond to my repair request?"
        }
    ],

    "I02": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "I've been told my rent is going up and I think the increase is too much. How much notice should I get and can I challenge it?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Rent increase seems too high. How much notice should I get and what can I do?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "My rents going up and i think its too high how much notice should they give and can i dispute it?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "If I disagree with the amount of a proposed rent increase, what notice is required and what options are available to challenge it?"
        }
    ],

    "I03": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "My landlord is trying to take money from my bond for damage that was already there. What can I use to show it wasn't caused by me?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "They're claiming my bond for old damage. What can I use to dispute it?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "My landlord wants my bond for damage that was already there what evidence can i use to dispute it?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "What information could help prove that damage being claimed against my bond existed before I moved in?"
        }
    ],

    "I04": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "Can my landlord evict me because I've used my rental rights, and what do they normally have to do to end my tenancy?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Can they kick me out for using my rights, and what do they need to end the rental?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "Can my landlord evict me cause i exercised my rental rights and what do they need to end the agreement?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "If I exercise my rights as a renter, can the rental provider end my agreement because of that, and what is normally required to end it?"
        }
    ],

    # -------------------------
    # Out-of-KB Questions
    # -------------------------

    "O01": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "Can I let someone else sublet my rental without asking my rental provider first?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Can I sublet without the landlord's permission?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "Can i sublet my place to someone else without asking the landlord?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "Do I need permission from my rental provider before another person can sublet the property?"
        }
    ],

    "O02": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "If I need to make a rental application to VCAT, how much will it cost me?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "How much does a VCAT rental application cost?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "How much dose it cost to make a rental application with VCAT?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "What fee would I have to pay to lodge a rental application with VCAT?"
        }
    ],

    "O03": [
        {
            "variant_id": "V1",
            "variant_type": "Conversational",
            "question": "Is my landlord allowed to put security cameras inside the place I'm renting?"
        },
        {
            "variant_id": "V2",
            "variant_type": "Short / informal",
            "question": "Can my landlord have cameras inside my rental?"
        },
        {
            "variant_id": "V3",
            "variant_type": "Grammar / spelling",
            "question": "Can my rental provider put security camras inside the property?"
        },
        {
            "variant_id": "V4",
            "variant_type": "Alternative structure",
            "question": "Are rental providers allowed to install indoor security cameras in a rented property?"
        }
    ]
}


# Convert variants to a simple DataFrame for review

variant_rows = []

for base_question_id, variants in question_variants.items():
    for variant in variants:
        variant_rows.append({
            "base_question_id": base_question_id,
            "variant_id": variant["variant_id"],
            "variant_type": variant["variant_type"],
            "question": variant["question"]
        })

variants_df = pd.DataFrame(variant_rows)

print(f"Base questions with variants: {variants_df['base_question_id'].nunique()}")
print(f"Total variants: {len(variants_df)}")

display(variants_df)


Base questions with variants: 15
Total variants: 60


,base_question_id,variant_id,variant_type,question
0,K01,V1,Conversational,"When I apply for a rental, can they ask if I h..."
1,K01,V2,Short / informal,Can they ask about problems I had with a previ...
2,K01,V3,Grammar / spelling,Can a landlord ask if ive had disputes with a ...
3,K01,V4,Alternative structure,If I had a dispute with a previous rental prov...
4,K02,V1,Conversational,"If I offer more than the advertised rent, is t..."
5,K02,V2,Short / informal,Can a landlord take an offer above the listed ...
6,K02,V3,Grammar / spelling,Can they accept more rent then what was advert...
7,K02,V4,Alternative structure,What happens if someone offers the rental prov...
8,K03,V1,Conversational,Does the place I'm renting in Victoria have to...
9,K03,V2,Short / informal,Does my rental need a fixed heater?


## 9. Build Robustness Test Collection

Each question variant represents the same underlying information need as its original question.

Rather than manually assigning the evaluation labels again, each variant inherits the original question's:

- question type
- relevant knowledge-base chunks
- expected answer

The original 15 questions are then combined with the 60 wording variants, creating a 75-question robustness test collection.


In [29]:
# Add original question labels to each wording variant

variants_labelled_df = variants_df.merge(
    test_collection_df[
        [
            "question_id",
            "question_type",
            "relevant_chunk_ids",
            "expected_answer"
        ]
    ],
    left_on="base_question_id",
    right_on="question_id",
    how="left"
)

# Create a unique ID for each variant
variants_labelled_df["question_id"] = (
    variants_labelled_df["base_question_id"]
    + "_"
    + variants_labelled_df["variant_id"]
)

# Keep columns in a clear order
variants_labelled_df = variants_labelled_df[
    [
        "question_id",
        "base_question_id",
        "variant_id",
        "variant_type",
        "question",
        "question_type",
        "relevant_chunk_ids",
        "expected_answer"
    ]
]


# Prepare the original 15 questions in the same format

original_questions_df = test_collection_df.copy()

original_questions_df["base_question_id"] = (
    original_questions_df["question_id"]
)

original_questions_df["variant_id"] = "Original"
original_questions_df["variant_type"] = "Original"

original_questions_df = original_questions_df[
    [
        "question_id",
        "base_question_id",
        "variant_id",
        "variant_type",
        "question",
        "question_type",
        "relevant_chunk_ids",
        "expected_answer"
    ]
]


# Combine original questions and wording variants

robustness_test_df = pd.concat(
    [
        original_questions_df,
        variants_labelled_df
    ],
    ignore_index=True
)


print(f"Original questions: {len(original_questions_df)}")
print(f"Question variants: {len(variants_labelled_df)}")
print(f"Total robustness questions: {len(robustness_test_df)}")

print("\nQuestions by type:")
print(
    robustness_test_df[
        "question_type"
    ].value_counts()
)

display(
    robustness_test_df.head(10)
)

Original questions: 15
Question variants: 60
Total robustness questions: 75

Questions by type:
question_type
Known        40
Inferred     20
Out-of-KB    15
Name: count, dtype: int64


,question_id,base_question_id,variant_id,variant_type,question,question_type,relevant_chunk_ids,expected_answer
0,K01,K01,Original,Original,Can a rental provider ask me about disputes wi...,Known,[RG_P07],No. A rental provider or agent cannot ask whet...
1,K02,K02,Original,Original,Can a rental provider accept an offer above th...,Known,[RG_P08],No. Rental providers and agents cannot ask for...
2,K03,K03,Original,Original,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]",Yes. Rental properties must have a fixed heate...
3,K04,K04,Original,Original,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]",You must return one completed and signed copy ...
4,K05,K05,Original,Original,How much rent can I be asked to pay in advance?,Known,[RG_P21],"If rent is paid weekly, you can be asked for u..."
5,K06,K06,Original,Original,How much notice must I receive before my rent ...,Known,[RG_P22],The rental provider must give you a Notice of ...
6,K07,K07,Original,Original,Is a broken toilet considered an urgent repair?,Known,[RG_P24],Yes. A blocked or broken toilet system is lega...
7,K08,K08,Original,Original,What happens if my rental provider wants to re...,Known,[RG_P28],If the rental provider wants to refuse consent...
8,I01,I01,Original,Original,My rental has no working fixed heater. Is this...,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]",Yes. A rental property must have a working fix...
9,I02,I02,Original,Original,My rent is being increased and I think the new...,Inferred,"[RG_P22, RG_P23]",You must receive a Notice of rent increase at ...


## 10. Validate Robustness Test Collection

Before saving the expanded test collection, run a set of checks to make sure the question variants were created correctly.

The validation confirms that:

- all 15 original questions are included
- each original question has exactly four variants
- all 75 question IDs are unique
- each variant keeps the same question type, relevant chunks and expected answer as its original question
- all labelled relevant chunks still exist in the knowledge base


In [30]:
# Validate the robustness test collection

validation_results = []


# 1. Expected collection sizes
validation_results.append({
    "check": "15 original questions",
    "passed": len(original_questions_df) == 15
})

validation_results.append({
    "check": "60 question variants",
    "passed": len(variants_labelled_df) == 60
})

validation_results.append({
    "check": "75 total questions",
    "passed": len(robustness_test_df) == 75
})


# 2. Every base question should have exactly four variants
variant_counts = (
    variants_labelled_df
    .groupby("base_question_id")
    .size()
)

validation_results.append({
    "check": "4 variants per base question",
    "passed": (
        len(variant_counts) == 15
        and (variant_counts == 4).all()
    )
})


# 3. Question IDs should all be unique
validation_results.append({
    "check": "Unique question IDs",
    "passed": robustness_test_df["question_id"].is_unique
})


# 4. Check inherited labels against original questions
base_lookup = (
    test_collection_df
    .set_index("question_id")
)

label_mismatches = []

for _, row in variants_labelled_df.iterrows():

    base_row = base_lookup.loc[
        row["base_question_id"]
    ]

    if (
        row["question_type"]
        != base_row["question_type"]
        or row["relevant_chunk_ids"]
        != base_row["relevant_chunk_ids"]
        or row["expected_answer"]
        != base_row["expected_answer"]
    ):
        label_mismatches.append(
            row["question_id"]
        )

validation_results.append({
    "check": "Variant labels match originals",
    "passed": len(label_mismatches) == 0
})


# 5. Check relevant chunk IDs against the KB
valid_chunk_ids = set(
    kb_df["chunk_id"]
)

invalid_chunks = []

for _, row in robustness_test_df.iterrows():

    for chunk_id in row["relevant_chunk_ids"]:

        if chunk_id not in valid_chunk_ids:
            invalid_chunks.append(
                (
                    row["question_id"],
                    chunk_id
                )
            )

validation_results.append({
    "check": "All relevant chunk IDs exist in KB",
    "passed": len(invalid_chunks) == 0
})


# Display validation results
validation_df = pd.DataFrame(
    validation_results
)

display(validation_df)


print("\nVariant counts per base question:")
display(
    variant_counts
    .rename("variant_count")
    .to_frame()
)


print("\nLabel mismatches:", label_mismatches)
print("Invalid chunk IDs:", invalid_chunks)

,check,passed
0,15 original questions,True
1,60 question variants,True
2,75 total questions,True
3,4 variants per base question,True
4,Unique question IDs,True
5,Variant labels match originals,True
6,All relevant chunk IDs exist in KB,True



Variant counts per base question:


,variant_count
base_question_id,
I01,4
I02,4
I03,4
I04,4
K01,4
K02,4
K03,4
K04,4
K05,4



Label mismatches: []
Invalid chunk IDs: []


## 11. Save Robustness Test Collection

Save the expanded 75-question test collection as a separate file so the original 15-question evaluation set remains unchanged.

This file will be used for the robustness evaluation of the final V2 RAG system.

In [31]:
# Save the expanded robustness test collection

robustness_output_path = (
    DATA_DIR / "rental_test_collection_robustness.jsonl"
)

robustness_test_df.to_json(
    robustness_output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved robustness test collection to:")
print(robustness_output_path)

print(f"\nQuestions saved: {len(robustness_test_df)}")
print(
    f"Base questions: "
    f"{robustness_test_df['base_question_id'].nunique()}"
)

Saved robustness test collection to:
/Users/seanrichards/Documents/University/DS Case Studies/Project/data/processed/rental_test_collection_robustness.jsonl

Questions saved: 75
Base questions: 15
